# Three ways to get R² ≈ 0.98 on air quality, and what the honest number is

UCI Air Quality: 9,357 hourly readings from a gas multisensor device. The standard
write-up of this dataset fits a linear model, reports an almost-perfect R², checks that
train and test scores are close, concludes there is no overfitting, and stops.

All of that is true and none of it means the model works. There are three separate leaks
available on this dataset, and this notebook measures what each one is worth.

The result that matters is not "leakage is bad". It is that the three leaks are **not the
same size of problem** — and that telling them apart takes a repeated measurement, not a
single run.

Everything below imports `src/airquality/`, which has no third-party dependency.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

from airquality.data import COLOCATED_SENSOR, TARGET, load_csv
from airquality.experiment import effects_summary, repeat, run, table, verdict
from airquality.linalg import NearestNeighbour
from airquality.split import chronological, gap, shuffled
from airquality.synthetic import generate

## Leak 1: the target is in the feature matrix under another name

The `PT08.S*` columns are tin-oxide sensors. Each one is **calibrated against the
co-located reference analyser** — which, for `PT08.S2(NMHC)`, is the benzene analyser
that produces the target `C6H6(GT)`.

They are two readings of the same physical quantity. A model given both is not predicting
benzene from meteorology and traffic pollutants; it is converting one calibrated reading
into another, and it will report R² ≈ 0.98 for doing so.

Nothing in the column names says this. It is in the dataset documentation, and it is the
entire result.

In [ ]:
print("target:", TARGET)
print("leak:  ", COLOCATED_SENSOR)

## Leak 2: a shuffled split on a time series

`train_test_split(..., shuffle=True)` is the reflex. On hourly sensor data it puts 14:00
in training and 15:00 in test, so the model interpolates between points it has already
seen rather than predicting anything.

The honest split is chronological — and it is the only arrangement that matches how the
model would actually be used.

In [ ]:
dataset = generate(2000, seed=0)  # synthetic: see below for why

for split in (chronological(dataset), shuffled(dataset)):
    print(
        f"{split.kind:<15}train/test {split.sizes[0]:>5}/{split.sizes[1]:<5}"
        f"leaks time: {split.leaks_time!s:<6}gap: {gap(split):+.0f}h"
    )

## Leak 3: fitting the scaler before splitting

The smallest of the three and the easiest to miss, because nothing about the resulting
numbers looks wrong. `features.standardise` takes train and test together and computes
everything from the training rows, so it cannot be called the other way by accident.

## Why a generated series, not the real one

On the real dataset you can show that removing the sensor lowers R². You cannot show what
the **correct** number is, so you cannot tell a good model from a leaking one.

The generator sets the signal-to-noise deliberately: the target is smooth and
autocorrelated, one feature is a noisy affine function of it (exactly what a calibrated
tin-oxide sensor is), and the rest carry a real but much weaker relationship. An honest
model scoring 0.98 here would mean the generator was leaking too — which makes it a test.

`--csv` runs the same experiment on the real download; see the README.

## The 2×2

Same model, same data, four arrangements.

In [ ]:
arrangements = run(dataset, seed=0) + run(
    dataset, model_factory=lambda: NearestNeighbour(k=1), seed=0
)
print(table(arrangements))

In [ ]:
print(verdict([a for a in arrangements if a.model == "LeastSquares"]))

## The part the first version of this experiment got wrong

The numbers above come from one run. The effect of a leak is itself a random variable —
it depends on which rows landed in the test set — so quoting a single number for it is
the same class of error as the leak itself: reporting an arrangement-specific artifact as
a property of the problem.

`repeat()` regenerates the series per run and reports the mean, the spread and the range.

In [ ]:
RUNS = 12
print(effects_summary(repeat(runs=RUNS, hours=2000), RUNS))

In [ ]:
# A memorising model exposes a split leak more than a linear one can.
# k-NN prediction is O(train x test), so this uses a shorter series.
print(
    effects_summary(
        repeat(runs=RUNS, hours=800, model_factory=lambda: NearestNeighbour(k=1)), RUNS
    )
)

Read the two rows carefully:

- **co-located sensor**: worth roughly +0.5 R², in *every* run, never close to zero.
- **shuffled split**: positive on average, and smaller than its own run-to-run spread —
  the range crosses zero, so a single experiment cannot establish its sign.

Both of these appear on the same checklist of things that invalidate a machine-learning
result. On this data one of them is decisive and the other is not separable from noise in
any one run. Which is which is a property of the data **and** the model's capacity to
memorise — note that the nearest-neighbour numbers are larger — so it is worth measuring
rather than assuming.

## Running it on the real dataset

```bash
curl -O https://archive.ics.uci.edu/static/public/360/air+quality.zip
unzip air+quality.zip
```

In [ ]:
csv = Path("AirQualityUCI.csv")
if csv.exists():
    real = load_csv(csv)
    print(f"rows {len(real):,}  columns {len(real.columns)}")
    for column in real.columns:
        print(f"  {column:<16}{real.missing_fraction(column):6.1%} missing")
    print()
    print(table(run(real, seed=0)))
else:
    print("AirQualityUCI.csv not found — see the README for the download.")

Three details in the loader that decide whether any of this is meaningful:

- **`-200` is the missing-value code.** Read as a number it is a plausible-looking value
  two orders of magnitude outside the real range, and a column that is 90% missing looks
  like a column with a strong negative signal.
- **`NMHC(GT)` is 90% absent and is dropped, not imputed.** Imputing it manufactures a
  column of one repeated estimate and gives a tree something to split on.
- **The file is semicolon-separated with decimal commas**, and ends with a block of empty
  rows that becomes an all-missing feature if it survives.

## Conclusion

The original exercise concluded that the polynomial model was best because train and test
errors were both low and close together. That check cannot detect any of the three leaks
here: under all of them, train and test scores stay close — the test set is contaminated,
not the fit.

What would have caught it:

1. Asking what each column physically **is**, before modelling. `PT08.S2(NMHC)` is a
   benzene sensor and `C6H6(GT)` is benzene.
2. Splitting a time series by time.
3. Repeating the comparison enough times to know which differences are real.

And the finding worth carrying: leakage warnings come as a flat checklist, and the items
on it differ in size by an order of magnitude. Ranking them takes a measurement.